# Test latency and real-time

## REQUIREMENTS

In [ ]:
!pip install -q transformers accelerate sentencepiece
!pip install -q joblib
!pip install -q bitsandbytes

In [1]:
deepseek_path="../../fine-tuned-model-attacks/deepseek_finetuned_model_multi"
xgboost_path="/home/jorge/TFM/ThreatLogLLM/ML-training/models/models_more_samples/xgboost_model.pkl"
dataset_path="/home/jorge/TFM/ThreatLogLLM/data/dataset-csv/DNN-EdgeIIoT-dataset.csv"
expected_columns_path ="/home/jorge/TFM/ThreatLogLLM/ML-training/models/models_more_samples/exports/xgb_input_columns.pkl"
label_encoder_path ="/home/jorge/TFM/ThreatLogLLM/ML-training/models/models_more_samples/exports/label_encoder.pkl"
attack_mapping_path ="/home/jorge/TFM/ThreatLogLLM/ML-training/models/models_more_samples/exports/attack_mapping.pkl"

## DEEPSEEK MEASURES

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    deepseek_path,
    fix_mistral_regex=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    deepseek_path,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=None,   
)

model.to(DEVICE)
model.eval()

print("Model and tokenizer loaded correctly!")

/home/jorge/TFM/threatlogllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


`low_cpu_mem_usage` was None, now default to True since model is quantized.


Model and tokenizer loaded correctly!


In [3]:
example_log_text = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request. Before answering, think carefully about the question "
    "and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n"
    "### Instruction:\n"
    "You are a cybersecurity expert specializing in IoT security. Your task is to analyze network logs and determine "
    "whether the given log data indicates a potential attack. Only provide the type of attack if it is an attack. "
    "If the log data is normal traffic, state that it is normal traffic.\n"
    "### Question:\n"
    "- The length of the DNS query is: 0.0\n"
    "- The MQTT protocol name used is: 0.0\n"
    "- The MQTT message type is: 0.0\n"
    "- The MQTT topic is: 0.0\n"
    "- The MQTT connection acknowledgment flags are: 0.0\n"
    "- TCP options set in the packet are: 0101080ade3913be0678be5f\n"
    "- TCP destination port is: 4321.0\n"
    "### Response:\n"
)
inputs_single = tokenizer(
    example_log_text,
    return_tensors="pt",
).to(DEVICE)


In [4]:
batch_size = 32
batch_texts = [example_log_text] * batch_size
inputs_batch = tokenizer(
    batch_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(DEVICE)


In [ ]:
import time
import numpy as np

def measure_llm_latency_single(num_runs=30, max_new_tokens=3):
    times = []

    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs_single, max_new_tokens=max_new_tokens)
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            _ = model.generate(**inputs_single, max_new_tokens=max_new_tokens)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        end = time.time()
        times.append((end - start) * 1000.0)  # ms

    times = np.array(times)
    print(f"[DeepSeek - single] Mean latency: {times.mean():.2f} ms")
    print(f"[DeepSeek - single] P95 latency: {np.percentile(times, 95):.2f} ms",end="\n\n")

def measure_llm_latency_batch(num_runs=20, max_new_tokens=3):
    times = []

    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs_batch, max_new_tokens=max_new_tokens)
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            _ = model.generate(**inputs_batch, max_new_tokens=max_new_tokens)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        end = time.time()
        times.append((end - start) * 1000.0)  # ms

    times = np.array(times)
    mean_batch = times.mean()
    mean_per_log = mean_batch / batch_size
    throughput = batch_size / (mean_batch / 1000.0)

    print(f"[DeepSeek - batch={batch_size}] Mean batch latency: {mean_batch:.2f} ms")
    print(f"[DeepSeek - batch={batch_size}] Mean latency/log: {mean_per_log:.2f} ms")
    print(f"[DeepSeek - batch={batch_size}] Approx throughput: {throughput:.1f} logs/s")

# Execute measurements
measure_llm_latency_single(max_new_tokens=3)
measure_llm_latency_batch(max_new_tokens=3)

[DeepSeek - single] Mean latency: 623.21 ms
[DeepSeek - single] P95 latency: 689.87 ms

[DeepSeek - batch=32] Mean batch latency: 63989.74 ms
[DeepSeek - batch=32] Mean latency/log: 1999.68 ms
[DeepSeek - batch=32] Approx throughput: 0.5 logs/s


## XGBOOST MEASURES

In [ ]:
import time
import numpy as np
import pandas as pd
import joblib

# Load model
xgb_model = joblib.load(xgboost_path)
print("XGBoost model loaded.")

# Load expected columns, label encoder and attack mapping
expected_columns = joblib.load(expected_columns_path)   
label_encoder = joblib.load(label_encoder_path)         
attack_mapping = joblib.load(attack_mapping_path)  

XGBoost model loaded.


In [ ]:
# Load and preprocess dataset
df = pd.read_csv(dataset_path, low_memory=False)

selected_columns = [
    "dns.qry.name.len", "mqtt.protoname", "mqtt.msg", "mqtt.topic",
    "mqtt.conack.flags", "tcp.options", "tcp.dstport", "Attack_type"
]
df_selected = df[selected_columns].dropna()

# Sample 21k random samples from the selected data
df_sampled = df_selected.sample(n=21000, random_state=42).reset_index(drop=True)

# Define X and y
X = df_sampled.drop(columns=["Attack_type"])
y_true = df_sampled["Attack_type"]

# Categorical encoding
categorical_columns = ["mqtt.protoname", "mqtt.msg", "mqtt.topic",
                       "mqtt.conack.flags", "tcp.options"]

X_encoded = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

# Same feature alignment as in training
missing_cols = [col for col in expected_columns if col not in X_encoded.columns]
missing_df = pd.DataFrame(0, index=X_encoded.index, columns=missing_cols)

X_encoded = pd.concat([X_encoded, missing_df], axis=1)
X_encoded = X_encoded[expected_columns] 

print("X_encoded shape:", X_encoded.shape)


X_encoded shape: (21000, 7462)


In [ ]:

# Select a single sample
X_sample = X_encoded.iloc[[0]].to_numpy()  

# For batch: repeat that row many times
batch_size_ml = 1024
X_batch = np.repeat(X_sample, batch_size_ml, axis=0) 

print("X_sample shape:", X_sample.shape)
print("X_batch shape :", X_batch.shape)



X_sample shape: (1, 7462)
X_batch shape : (1024, 7462)


In [ ]:
# Measure XGBoost latency
def measure_xgb_latency_single(num_runs=200):
    times = []
    for _ in range(num_runs):
        start = time.time()
        _ = xgb_model.predict(X_sample)
        end = time.time()
        times.append((end - start) * 1000.0)  # ms
    times = np.array(times)
    print(f"[XGBoost - single] Mean latency: {times.mean():.4f} ms")
    print(f"[XGBoost - single] P95 latency: {np.percentile(times, 95):.4f} ms",end="\n\n")

def measure_xgb_latency_batch(num_runs=50):
    times = []
    for _ in range(num_runs):
        start = time.time()
        _ = xgb_model.predict(X_batch)
        end = time.time()
        times.append((end - start) * 1000.0)  # ms
    times = np.array(times)
    mean_batch = times.mean()
    mean_per_log = mean_batch / batch_size_ml
    throughput = batch_size_ml / (mean_batch / 1000.0)
    print(f"[XGBoost - batch={batch_size_ml}] Mean batch latency: {mean_batch:.4f} ms")
    print(f"[XGBoost - batch={batch_size_ml}] Mean latency/log: {mean_per_log:.6f} ms")
    print(f"[XGBoost - batch={batch_size_ml}] Approx throughput: {throughput:.1f} logs/s")

# Execute measurements
measure_xgb_latency_single()
measure_xgb_latency_batch()

[XGBoost - single] Mean latency: 1.0994 ms
[XGBoost - single] P95 latency: 3.6321 ms

[XGBoost - batch=1024] Mean batch latency: 227.2748 ms
[XGBoost - batch=1024] Mean latency/log: 0.221948 ms
[XGBoost - batch=1024] Approx throughput: 4505.6 logs/s
